In [1]:
from pathlib import Path
import hashlib
import pandas as pd

print("02.25 STEP 1 - INSPECT BASELINE FILES")
print("=" * 65)

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

print(f"\nHistorical sales files found: {len(sales_files)}")


def calculate_file_hash(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


file_records = []

for file in sales_files:
    file_records.append({
        "file_name": file.name,
        "file_size_bytes": file.stat().st_size,
        "sha256_hash": calculate_file_hash(file)
    })

baseline_df = pd.DataFrame(file_records)

print("\nBASELINE FILE INFORMATION")
print("-" * 65)

print(baseline_df.to_string(index=False))

print("\nHASH DUPLICATE CHECK")
print("-" * 65)

duplicate_hashes = baseline_df[
    baseline_df["sha256_hash"].duplicated(keep=False)
]

print(f"Files with duplicate hashes: {len(duplicate_hashes)}")

if len(duplicate_hashes) > 0:
    print("\nDuplicate hash groups:")
    print(duplicate_hashes.to_string(index=False))
else:
    print("No duplicate file hashes found.")

print("\n02.25 Step 1 inspection: COMPLETED")

02.25 STEP 1 - INSPECT BASELINE FILES

Historical sales files found: 32

BASELINE FILE INFORMATION
-----------------------------------------------------------------
        file_name  file_size_bytes                                                      sha256_hash
sales_2024-01.csv          7748000 7e15991caf2f227df332e32c23e734a7212e130279fecdd32f28c77565c0fd54
sales_2024-02.csv          7339383 ce10136c4146732de96acd1f659dd4df2e26b6147d54e2aaf107d84142b554cb
sales_2024-03.csv          8155637 8492bd2af496b212bbc429bdef79ce58472a1e3416cf57a88f837d5dd96db59b
sales_2024-04.csv          8316188 c8d3264b87ffb3d3004ac9ac76ff153f2b30cd066a4f789a9c693533f5392b54
sales_2024-05.csv          8562591 2a753a5cffefd5b22ea8b6a91c5a7bc4df176e6a092fde48ad846064f83b845f
sales_2024-06.csv          8811819 8c59e97a9bea31240eb167249614040ca2e9495d3f855dec1df3391f286e67f0
sales_2024-07.csv          8402019 289866871be589141920515c830503dd3029f8cdb6146747c08cb2c92fd754eb
sales_2024-08.csv          8811337 

In [2]:
from pathlib import Path
import shutil
import hashlib

print("02.25 STEP 2 - CREATE EXACT DUPLICATE FILE")
print("=" * 65)

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

OUTPUT_DIR = SALES_DIR / "exact_duplicate_file_adjusted"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Source file selected for the duplicate scenario
SOURCE_FILE = SALES_DIR / "sales_2025-06.csv"

# Duplicate file name
DUPLICATE_FILE = OUTPUT_DIR / "sales_2025-06_DUPLICATE.csv"


def calculate_file_hash(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


print(f"\nSource file:")
print(SOURCE_FILE)

print(f"\nDuplicate file:")
print(DUPLICATE_FILE)

# Create exact byte-for-byte copy
shutil.copy2(SOURCE_FILE, DUPLICATE_FILE)

# Calculate hashes
source_hash = calculate_file_hash(SOURCE_FILE)
duplicate_hash = calculate_file_hash(DUPLICATE_FILE)

print("\nDUPLICATE FILE VALIDATION")
print("-" * 65)

print(f"Source file size: {SOURCE_FILE.stat().st_size:,} bytes")
print(f"Duplicate file size: {DUPLICATE_FILE.stat().st_size:,} bytes")

print(f"\nSource SHA-256:")
print(source_hash)

print(f"\nDuplicate SHA-256:")
print(duplicate_hash)

hash_match = source_hash == duplicate_hash

print(f"\nExact hash match: {hash_match}")

if hash_match:
    print("\n02.25 Step 2 exact duplicate creation: PASSED")
else:
    print("\n02.25 Step 2 exact duplicate creation: FAILED")

02.25 STEP 2 - CREATE EXACT DUPLICATE FILE

Source file:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\sales_2025-06.csv

Duplicate file:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\exact_duplicate_file_adjusted\sales_2025-06_DUPLICATE.csv

DUPLICATE FILE VALIDATION
-----------------------------------------------------------------
Source file size: 9,560,175 bytes
Duplicate file size: 9,560,175 bytes

Source SHA-256:
0bb4e3539d862c90ae594b8c7d38685685973733f0a6b33ee6d4fc492c90083c

Duplicate SHA-256:
0bb4e3539d862c90ae594b8c7d38685685973733f0a6b33ee6d4fc492c90083c

Exact hash match: True

02.25 Step 2 exact duplicate creation: PASSED


In [3]:
from pathlib import Path
import hashlib
import pandas as pd

print("02.25 STEP 3 - VALIDATE EXACT DUPLICATE FILE")
print("=" * 65)

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"
OUTPUT_DIR = SALES_DIR / "exact_duplicate_file_adjusted"

SOURCE_FILE = SALES_DIR / "sales_2025-06.csv"
DUPLICATE_FILE = OUTPUT_DIR / "sales_2025-06_DUPLICATE.csv"


def calculate_file_hash(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


source_df = pd.read_csv(SOURCE_FILE)
duplicate_df = pd.read_csv(DUPLICATE_FILE)

source_hash = calculate_file_hash(SOURCE_FILE)
duplicate_hash = calculate_file_hash(DUPLICATE_FILE)

row_count_same = len(source_df) == len(duplicate_df)
schema_same = list(source_df.columns) == list(duplicate_df.columns)
size_same = SOURCE_FILE.stat().st_size == DUPLICATE_FILE.stat().st_size
hash_same = source_hash == duplicate_hash

print("\nDUPLICATE VALIDATION")
print("-" * 65)

print(f"Source rows: {len(source_df):,}")
print(f"Duplicate rows: {len(duplicate_df):,}")

print(f"\nSource columns: {len(source_df.columns)}")
print(f"Duplicate columns: {len(duplicate_df.columns)}")

print(f"\nFile size same: {size_same}")
print(f"Schema same: {schema_same}")
print(f"Row count same: {row_count_same}")
print(f"SHA-256 hash same: {hash_same}")

print("\nFINAL VALIDATION")
print("-" * 65)

if (
    size_same
    and schema_same
    and row_count_same
    and hash_same
):
    print("Exact duplicate confirmed: True")
    print("\n02.25 Step 3 validation: PASSED")
else:
    print("Exact duplicate confirmed: False")
    print("\n02.25 Step 3 validation: FAILED")

02.25 STEP 3 - VALIDATE EXACT DUPLICATE FILE

DUPLICATE VALIDATION
-----------------------------------------------------------------
Source rows: 65,300
Duplicate rows: 65,300

Source columns: 15
Duplicate columns: 15

File size same: True
Schema same: True
Row count same: True
SHA-256 hash same: True

FINAL VALIDATION
-----------------------------------------------------------------
Exact duplicate confirmed: True

02.25 Step 3 validation: PASSED


In [4]:
from pathlib import Path
import hashlib
import pandas as pd

print("02.25 STEP 4 - FINAL EXACT DUPLICATE FILE VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"
OUTPUT_DIR = SALES_DIR / "exact_duplicate_file_adjusted"

SOURCE_FILE = SALES_DIR / "sales_2025-06.csv"
DUPLICATE_FILE = OUTPUT_DIR / "sales_2025-06_DUPLICATE.csv"

VALIDATION_DIR = SALES_DIR / "exact_duplicate_file_validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)


def file_hash(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


source_df = pd.read_csv(SOURCE_FILE)
duplicate_df = pd.read_csv(DUPLICATE_FILE)

source_hash = file_hash(SOURCE_FILE)
duplicate_hash = file_hash(DUPLICATE_FILE)

validation = {
    "source_file": SOURCE_FILE.name,
    "duplicate_file": DUPLICATE_FILE.name,
    "source_rows": len(source_df),
    "duplicate_rows": len(duplicate_df),
    "source_columns": len(source_df.columns),
    "duplicate_columns": len(duplicate_df.columns),
    "source_file_size_bytes": SOURCE_FILE.stat().st_size,
    "duplicate_file_size_bytes": DUPLICATE_FILE.stat().st_size,
    "row_count_same": len(source_df) == len(duplicate_df),
    "schema_same": list(source_df.columns) == list(duplicate_df.columns),
    "file_size_same": SOURCE_FILE.stat().st_size == DUPLICATE_FILE.stat().st_size,
    "sha256_same": source_hash == duplicate_hash,
    "exact_duplicate_confirmed": (
        len(source_df) == len(duplicate_df)
        and list(source_df.columns) == list(duplicate_df.columns)
        and SOURCE_FILE.stat().st_size == DUPLICATE_FILE.stat().st_size
        and source_hash == duplicate_hash
    )
}

print("\nFINAL VALIDATION")
print("-" * 65)

for key, value in validation.items():
    print(f"{key}: {value}")

validation_df = pd.DataFrame([validation])

output_file = VALIDATION_DIR / "exact_duplicate_file_validation.csv"
validation_df.to_csv(output_file, index=False)

print("\nValidation summary saved to:")
print(output_file)

if validation["exact_duplicate_confirmed"]:
    print("\n02.25 Step 4 validation: PASSED")
else:
    print("\n02.25 Step 4 validation: FAILED")

02.25 STEP 4 - FINAL EXACT DUPLICATE FILE VALIDATION

FINAL VALIDATION
-----------------------------------------------------------------
source_file: sales_2025-06.csv
duplicate_file: sales_2025-06_DUPLICATE.csv
source_rows: 65300
duplicate_rows: 65300
source_columns: 15
duplicate_columns: 15
source_file_size_bytes: 9560175
duplicate_file_size_bytes: 9560175
row_count_same: True
schema_same: True
file_size_same: True
sha256_same: True
exact_duplicate_confirmed: True

Validation summary saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\exact_duplicate_file_validation\exact_duplicate_file_validation.csv

02.25 Step 4 validation: PASSED
